In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for UTC+8
shanghai = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')

# Convert epoch seconds to timezone-aware datetimes in Asia/Shanghai
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)

# Define the time window in UTC+8
window_start = pd.Timestamp('2020-05-29 02:00:00', tz=shanghai)
window_end = pd.Timestamp('2020-05-29 02:30:00', tz=shanghai)

# Total rows
metric_total_rows = metric_df.shape[0]
trace_total_rows = trace_df.shape[0]

# --- Aggregation for metric.csv ---
# Compute global P95 per (cmdb_id, kpi_name) using the full series (rule 9)
metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False)

metric_agg = metric_grp.agg(
    total_points = ('value', 'size'),
    min_timestamp = ('dt', 'min'),
    max_timestamp = ('dt', 'max'),
)

# compute P95 separately to ensure correct quantile behavior
metric_p95 = metric_grp['value'].quantile(0.95).reset_index().rename(columns={'value':'global_P95_value'})

metric_agg = metric_agg.merge(metric_p95, on=['cmdb_id','kpi_name'], how='left')

# Count points in the specified window
metric_in_window = metric_df[(metric_df['dt'] >= window_start) & (metric_df['dt'] <= window_end)]
metric_window_counts = metric_in_window.groupby(['cmdb_id','kpi_name'], as_index=False).size().rename(columns={'size':'points_in_window'})

metric_agg = metric_agg.merge(metric_window_counts, on=['cmdb_id','kpi_name'], how='left')
metric_agg['points_in_window'] = metric_agg['points_in_window'].fillna(0).astype(int)

# Format timestamps as ISO strings for compact display and round P95
metric_agg['min_timestamp'] = metric_agg['min_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
metric_agg['max_timestamp'] = metric_agg['max_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
metric_agg['global_P95_value'] = metric_agg['global_P95_value'].round(6)

# Keep columns in requested order and limit to top 20 rows
metric_display = metric_agg[['cmdb_id','kpi_name','total_points','min_timestamp','max_timestamp','global_P95_value','points_in_window']].head(20)

# --- Aggregation for trace.csv ---
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False)

trace_agg = trace_grp.agg(
    total_points = ('value', 'size'),
    min_timestamp = ('dt', 'min'),
    max_timestamp = ('dt', 'max'),
)

trace_p95 = trace_grp['value'].quantile(0.95).reset_index().rename(columns={'value':'global_P95_value'})

trace_agg = trace_agg.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')

trace_in_window = trace_df[(trace_df['dt'] >= window_start) & (trace_df['dt'] <= window_end)]
trace_window_counts = trace_in_window.groupby(['cmdb_id','trace_name'], as_index=False).size().rename(columns={'size':'points_in_window'})

trace_agg = trace_agg.merge(trace_window_counts, on=['cmdb_id','trace_name'], how='left')
trace_agg['points_in_window'] = trace_agg['points_in_window'].fillna(0).astype(int)

trace_agg['min_timestamp'] = trace_agg['min_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
trace_agg['max_timestamp'] = trace_agg['max_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
trace_agg['global_P95_value'] = trace_agg['global_P95_value'].round(6)

trace_display = trace_agg[['cmdb_id','trace_name','total_points','min_timestamp','max_timestamp','global_P95_value','points_in_window']].head(20)

# Display compact results: metric_display, metric_total_rows, trace_display, trace_total_rows
metric_display, metric_total_rows, trace_display, trace_total_rows

```
Out[1]:
```
```python
# Summarize the previously computed aggregates in plain English
metric_groups_shown = metric_display.shape[0]
metric_window_groups = (metric_display['points_in_window'] > 0).sum()
metric_window_points_total = int(metric_display['points_in_window'].sum())
metric_total_points_range = (int(metric_display['total_points'].min()), int(metric_display['total_points'].max()))
metric_earliest = metric_display['min_timestamp'].min()
metric_latest = metric_display['max_timestamp'].max()

trace_groups_shown = trace_display.shape[0]
trace_window_groups = (trace_display['points_in_window'] > 0).sum()
trace_window_points_total = int(trace_display['points_in_window'].sum())
trace_total_points_range = (int(trace_display['total_points'].min()), int(trace_display['total_points'].max()))
trace_earliest = trace_display['min_timestamp'].min()
trace_latest = trace_display['max_timestamp'].max()

summary = (
    f"Summary:\n"
    f"- metric.csv: total rows = {metric_total_rows}. Aggregated by (cmdb_id, kpi_name); showing {metric_groups_shown} groups (limited to 20).\n"
    f"  For the displayed groups (mostly cmdb_id=db_001): total_points range = {metric_total_points_range}. "
    f"{metric_window_groups} groups have ≥1 point inside the window 2020-05-29 02:00:00 to 2020-05-29 02:30:00 (Asia/Shanghai), "
    f"total points in-window among displayed = {metric_window_points_total}. "
    f"Earliest min_timestamp (displayed) = {metric_earliest}; latest max_timestamp (displayed) = {metric_latest}.\n\n"
    f"- trace.csv: total rows = {trace_total_rows}. Aggregated by (cmdb_id, trace_name); showing {trace_groups_shown} groups (limited to 20).\n"
    f"  For the displayed groups (all appear to be cmdb_id=docker_001): total_points range = {trace_total_points_range}. "
    f"{trace_window_groups} groups have ≥1 point inside the same window; total points in-window among displayed = {trace_window_points_total}. "
    f"Earliest min_timestamp (displayed) = {trace_earliest}; latest max_timestamp (displayed) = {trace_latest}.\n\n"
    f"Notes: P95 values were computed over each group's full series; the time window referenced is in UTC+8 (Asia/Shanghai)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id             kpi_name  total_points             min_timestamp             max_timestamp  global_P95_value  points_in_window
0   db_001                  ACS            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          0.020000                30
1   db_001                 AIOS            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          0.010000                30
2   db_001                  AWS            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          0.020000                29
3   db_001          Asm_Free_Tb            12  2020-05-29 01:32:00+0800  2020-05-29 02:27:00+0800         12.510000                 6
4   db_001         CPU_Used_Pct             2  2020-05-29 01:37:00+0800  2020-05-29 02:00:00+0800          0.963954                 1
5   db_001         CPU_free_pct            59  2020-05-29 01:30:00+0800  2020-05-29 02:30:00+0800         99.361838                29
6   db_001         Call_Per_Sec            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800         10.200000                29
7   db_001  DFParaWrite_Per_Sec            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          2.270500                30
8   db_001               DbTime            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800         61.360000                29
9   db_001         Exec_Per_Sec            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800        262.320000                29
10  db_001                 Hang            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          1.000000                29
11  db_001  LFParaWrite_Per_Sec            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          2.335000                29
12  db_001       LFSync_Per_Sec            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          2.439500                30
13  db_001   Logic_Read_Per_Sec            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800       2336.810000                29
14  db_001        Login_Per_Sec            59  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800          0.300000                29
15  db_001            MEM_Total            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800        381.902264                30
16  db_001             MEM_Used            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800        246.825361                30
17  db_001         MEM_Used_Pct            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800         64.627777                30
18  db_001        MEM_real_util            60  2020-05-29 01:30:00+0800  2020-05-29 02:29:00+0800         46.380500                30
19  db_001      New_Tbs_Free_Gb             6  2020-05-29 01:37:00+0800  2020-05-29 02:27:00+0800        283.078691                 3, 71439,        cmdb_id                                     trace_name  total_points             min_timestamp             max_timestamp  global_P95_value  points_in_window
0   docker_001  trace.from_os_022.remoteprocess.duration_mean            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.464565                30
1   docker_001   trace.from_os_022.remoteprocess.duration_p95            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          1.619310                30
2   docker_001     trace.from_os_022.remoteprocess.error_rate            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.000000                30
3   docker_001      trace.from_os_022.remoteprocess.row_count            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800        136.500000                30
4   docker_001                   trace.self.csf.duration_mean            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.048389                30
5   docker_001                    trace.self.csf.duration_p95            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.117485                30
6   docker_001                      trace.self.csf.error_rate            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.000000                30
7   docker_001                       trace.self.csf.row_count            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800        546.000000                30
8   docker_001             trace.self.flyremote.duration_mean            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.009655                30
9   docker_001              trace.self.flyremote.duration_p95            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.016902                30
10  docker_001                trace.self.flyremote.error_rate            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.000000                30
11  docker_001                 trace.self.flyremote.row_count            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800        136.500000                30
12  docker_001                  trace.self.jdbc.duration_mean            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.003168                30
13  docker_001                   trace.self.jdbc.duration_p95            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.003000                30
14  docker_001                     trace.self.jdbc.error_rate            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.000000                30
15  docker_001                      trace.self.jdbc.row_count            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800       2184.450000                30
16  docker_001                 trace.self.local.duration_mean            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.010710                30
17  docker_001                  trace.self.local.duration_p95            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.090650                30
18  docker_001                    trace.self.local.error_rate            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800          0.000000                30
19  docker_001                     trace.self.local.row_count            30  2020-05-29 02:00:00+0800  2020-05-29 02:29:00+0800       1911.000000                30, 5524)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing metric_df and timezone if present; otherwise load and prepare
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

shanghai = pytz.timezone('Asia/Shanghai')

# ensure dt column exists and is timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)
else:
    # ensure dtype is datetime and timezone-aware
    if metric_df['dt'].dtype == object:
        metric_df['dt'] = pd.to_datetime(metric_df['dt']).dt.tz_localize(shanghai, ambiguous='NaT', nonexistent='NaT')

# define window (Asia/Shanghai)
window_start = pd.Timestamp('2020-05-29 02:00:00', tz=shanghai)
window_end = pd.Timestamp('2020-05-29 02:30:00', tz=shanghai)

# 1) Compute global P95 per group using full series (rule 9)
p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_P95_value'})

# 2) Total points per group (full series)
total_points_df = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False).size().rename(columns={'size':'total_points'})

# 3) Filter to window and prepare anomaly flags by merging global P95 into window points
window_df = metric_df[(metric_df['dt'] >= window_start) & (metric_df['dt'] <= window_end)].copy()
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# mark anomalies: value > global_P95_value
window_with_p95['is_anomaly'] = window_with_p95['value'] > window_with_p95['global_P95_value']

# 4) Aggregate window-level stats per group
window_agg = window_with_p95.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    points_in_window = ('value','size'),
    anomaly_count_in_window = ('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window = ('dt', lambda s: s[window_with_p95.loc[s.index,'is_anomaly']].min() if s.size>0 else pd.NaT),
    max_value_in_window = ('value','max')
)

# Note: the lambda above uses index alignment to find earliest anomaly per group's window slice.

# 5) Combine totals, p95, and window aggregates to produce one row per group
result = total_points_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left').merge(
    window_agg, on=['cmdb_id','kpi_name'], how='left'
)

# fill missing window-derived fields for groups with no points in window
result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
result['anomaly_count_in_window'] = result['anomaly_count_in_window'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window']  # leave NaN if no points
# earliest anomaly timestamp: convert NaT to None, otherwise format as string with timezone
def format_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(shanghai).strftime('%Y-%m-%d %H:%M:%S%z')

result['earliest_anomaly_timestamp_in_window'] = result['earliest_anomaly_timestamp_in_window'].apply(format_ts)

# Round P95 for compactness
result['global_P95_value'] = result['global_P95_value'].round(6)

# 6) Sort by anomaly_count_in_window desc, limit to top 20 rows
result_sorted = result.sort_values(['anomaly_count_in_window','points_in_window'], ascending=[False, False]).reset_index(drop=True)
results_display = result_sorted[['cmdb_id','kpi_name','total_points','points_in_window','global_P95_value',
                                 'anomaly_count_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window']].head(20)

# 7) Total number of metric anomaly points across all groups within the window (single scalar)
# Use the is_anomaly flags in window_with_p95 (covers only points in window with proper p95 mapping)
total_anomalies = int(window_with_p95['is_anomaly'].sum())

# Display compact outputs
results_display, total_anomalies

```
Out[2]:
```
```python
# Summarize the anomaly aggregation results in plain English using existing variables
try:
    result_sorted
    results_display
    total_anomalies
except NameError:
    # fallback: reload results by re-running the previous aggregation (should not be necessary in the kernel stateful session)
    import pandas as pd, pytz
    metric_df = pd.read_csv('metric.csv')
    shanghai = pytz.timezone('Asia/Shanghai')
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)
    window_start = pd.Timestamp('2020-05-29 02:00:00', tz=shanghai)
    window_end = pd.Timestamp('2020-05-29 02:30:00', tz=shanghai)
    p95_df = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_P95_value'})
    total_points_df = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False).size().rename(columns={'size':'total_points'})
    window_df = metric_df[(metric_df['dt'] >= window_start) & (metric_df['dt'] <= window_end)].copy()
    window_with_p95 = window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
    window_with_p95['is_anomaly'] = window_with_p95['value'] > window_with_p95['global_P95_value']
    window_agg = window_with_p95.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        points_in_window = ('value','size'),
        anomaly_count_in_window = ('is_anomaly','sum'),
        earliest_anomaly_timestamp_in_window = ('dt', lambda s: s[window_with_p95.loc[s.index,'is_anomaly']].min() if s.size>0 else pd.NaT),
        max_value_in_window = ('value','max')
    )
    result = total_points_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left').merge(window_agg, on=['cmdb_id','kpi_name'], how='left')
    result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
    result['anomaly_count_in_window'] = result['anomaly_count_in_window'].fillna(0).astype(int)
    result_sorted = result.sort_values(['anomaly_count_in_window','points_in_window'], ascending=[False, False]).reset_index(drop=True)
    results_display = result_sorted[['cmdb_id','kpi_name','total_points','points_in_window','global_P95_value',
                                     'anomaly_count_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window']].head(20)
    total_anomalies = int(window_with_p95['is_anomaly'].sum())

# Build concise top-5 summary lines
top_n = 5
top_rows = result_sorted[result_sorted['anomaly_count_in_window']>0].head(top_n)
top_lines = []
for _, r in top_rows.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    anom = int(r['anomaly_count_in_window'])
    pts = int(r['points_in_window'])
    earliest = r['earliest_anomaly_timestamp_in_window']
    if pd.isna(earliest):
        earliest_str = "none"
    else:
        # ensure string formatting if Timestamp
        try:
            earliest_str = pd.to_datetime(earliest).tz_convert(pytz.timezone('Asia/Shanghai')).strftime('%Y- %m-%d %H:%M:%S%z')
        except Exception:
            earliest_str = str(earliest)
    maxv = r['max_value_in_window']
    top_lines.append(f"{cmdb} / {kpi}: anomalies={anom}, points_in_window={pts}, earliest_anomaly={earliest_str}, max_in_window={maxv}")

num_groups_with_anomalies = int((result_sorted['anomaly_count_in_window'] > 0).sum())

summary = (
    f"Metric anomalies summary (window 2020-05-29 02:00:00 to 02:30:00 Asia/Shanghai):\n"
    f"- Total anomaly points across all metric groups within the window: {int(total_anomalies)}\n"
    f"- Number of metric groups with ≥1 anomaly in-window: {num_groups_with_anomalies}\n"
    f"- Top {min(top_n, num_groups_with_anomalies)} groups by anomaly count (examples):\n  - " + "\n  - ".join(top_lines)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                  kpi_name  total_points  points_in_window  global_P95_value  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
0   os_015  Incoming_network_traffic            61                31          0.002379                        3             2020-05-29 02:12:00+0800             0.003528
1   db_001             MEM_real_util            60                30         46.380500                        3             2020-05-29 02:02:00+0800            46.520000
2   db_005                  MEM_Used            60                30        430.075016                        3             2020-05-29 02:27:00+0800           430.129381
3   db_010              CPU_free_pct            60                30         97.892011                        3             2020-05-29 02:04:00+0800            98.002566
4   db_013                       ACS            60                30          0.010500                        3             2020-05-29 02:09:00+0800             0.020000
5   os_006  Outgoing_network_traffic            60                30          0.000134                        3             2020-05-29 02:16:00+0800             0.000147
6   os_007             Disk_avgqu_sz            60                30          0.001503                        3             2020-05-29 02:02:00+0800             0.002004
7   os_008                Disk_svctm            60                30          1.575000                        3             2020-05-29 02:06:00+0800             5.000000
8   os_011               Disk_wr_kbs            60                30          0.060581                        3             2020-05-29 02:01:00+0800             0.062120
9   os_011  Outgoing_network_traffic            60                30          0.010351                        3             2020-05-29 02:08:00+0800             0.028861
10  os_012             CPU_user_time            60                30          0.542835                        3             2020-05-29 02:13:00+0800             0.930959
11  os_012               Disk_wr_kbs            60                30          0.061637                        3             2020-05-29 02:10:00+0800             0.062244
12  os_012          Memory_available            60                30          1.722702                        3             2020-05-29 02:17:00+0800             1.723423
13  os_012  Outgoing_network_traffic            60                30          0.004005                        3             2020-05-29 02:09:00+0800             0.005908
14  os_013                Disk_svctm            60                30          2.093500                        3             2020-05-29 02:08:00+0800             5.850000
15  os_013  Incoming_network_traffic            60                30          0.003496                        3             2020-05-29 02:06:00+0800             0.008479
16  os_013               Memory_used            60                30          7.025187                        3             2020-05-29 02:20:00+0800             7.025547
17  os_013  Outgoing_network_traffic            60                30          0.005571                        3             2020-05-29 02:05:00+0800             0.008056
18  os_015                Disk_await            60                30          1.762000                        3             2020-05-29 02:06:00+0800             2.330000
19  os_016  Outgoing_network_traffic            60                30          0.001246                        3             2020-05-29 02:08:00+0800             0.001822, 1139)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing metric_df and timezone variables if present; otherwise load and prepare
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

shanghai = pytz.timezone('Asia/Shanghai')

# ensure dt exists and is timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)
else:
    # if exists but not tz-aware, convert
    if not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
        metric_df['dt'] = pd.to_datetime(metric_df['dt'], utc=True).dt.tz_convert(shanghai)

# Define incident window (Asia/Shanghai)
window_start = pd.Timestamp('2020-05-29 02:00:00', tz=shanghai)
window_end = pd.Timestamp('2020-05-29 02:30:00', tz=shanghai)

# 1) Recompute global P95 per (cmdb_id, kpi_name) over full series (must be done BEFORE filtering)
p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_P95_value'})

# 2) Filter metric points to the incident window
window_df = metric_df[(metric_df['dt'] >= window_start) & (metric_df['dt'] <= window_end)].copy()
# sort for consecutive detection
window_df = window_df.sort_values(['cmdb_id','kpi_name','dt']).reset_index(drop=True)

# 3) Merge global P95 into window points and mark anomalies
window_df = window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
window_df['is_anomaly'] = window_df['value'] > window_df['global_P95_value']

# 4) For each group find consecutive runs of anomalous points
def extract_runs(g):
    # g is sorted by dt
    if g.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
            'run_length_minutes','max_value_in_run','global_P95_value'
        ])
    mask = g['is_anomaly'].astype(int)
    # identify segment id that increments whenever mask changes
    seg = (mask != mask.shift(fill_value=0)).cumsum()
    g = g.assign(_seg=seg)
    # select only anomaly segments (mask == 1)
    anomalous = g[g['is_anomaly']]
    if anomalous.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
            'run_length_minutes','max_value_in_run','global_P95_value'
        ])
    agg = anomalous.groupby('_seg').agg(
        cmdb_id = ('cmdb_id','first'),
        kpi_name = ('kpi_name','first'),
        fault_start_timestamp = ('dt','min'),
        fault_end_timestamp = ('dt','max'),
        run_length_minutes = ('dt', lambda s: s.count()),  # count of consecutive points = minutes (per-minute series)
        max_value_in_run = ('value','max'),
        global_P95_value = ('global_P95_value','first')
    ).reset_index(drop=True)
    return agg

runs_list = []
grouped = window_df.groupby(['cmdb_id','kpi_name'], sort=False)
for _, g in grouped:
    runs = extract_runs(g)
    if not runs.empty:
        runs_list.append(runs)
if runs_list:
    runs_df = pd.concat(runs_list, ignore_index=True)
else:
    runs_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'run_length_minutes','max_value_in_run','global_P95_value'
    ])

# 4.5) Compute breach_ratio per run
runs_df['global_P95_value'] = runs_df['global_P95_value'].astype(float)
runs_df['max_value_in_run'] = runs_df['max_value_in_run'].astype(float)
runs_df['breach_ratio'] = (runs_df['max_value_in_run'] - runs_df['global_P95_value']) / runs_df['global_P95_value'].clip(lower=1e-9)

# 5) Filter runs to keep those with run_length_minutes >=2 OR breach_ratio >= 0.5
filtered_runs = runs_df[(runs_df['run_length_minutes'] >= 2) | (runs_df['breach_ratio'] >= 0.5)].copy()

# Format timestamps for compact display (Asia/Shanghai)
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(shanghai).strftime('%Y-%m-%d %H:%M:%S%z')

if not filtered_runs.empty:
    filtered_runs['fault_start_timestamp'] = filtered_runs['fault_start_timestamp'].apply(fmt_ts)
    filtered_runs['fault_end_timestamp'] = filtered_runs['fault_end_timestamp'].apply(fmt_ts)

# Round numeric columns for compactness
filtered_runs['global_P95_value'] = filtered_runs['global_P95_value'].round(6)
filtered_runs['max_value_in_run'] = filtered_runs['max_value_in_run'].round(6)
filtered_runs['breach_ratio'] = filtered_runs['breach_ratio'].round(6)

# 6) Return compact table limited to top 20 faults sorted by run_length_minutes desc, then breach_ratio desc
faults_sorted = filtered_runs.sort_values(['run_length_minutes','breach_ratio'], ascending=[False, False]).reset_index(drop=True)
faults_display = faults_sorted[['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                                'run_length_minutes','max_value_in_run','global_P95_value','breach_ratio']].head(20)

# Scalars to return
total_number_of_faults_identified = int(faults_sorted.shape[0])
number_of_unique_components_with_faults = int(faults_sorted['cmdb_id'].nunique()) if total_number_of_faults_identified>0 else 0
if total_number_of_faults_identified>0:
    # earliest start timestamp among identified faults (formatted)
    earliest_fault_start_timestamp = faults_sorted['fault_start_timestamp'].min()
else:
    earliest_fault_start_timestamp = None

# Display compact outputs (DataFrame and three scalars)
faults_display, total_number_of_faults_identified, number_of_unique_components_with_faults, earliest_fault_start_timestamp

```
Out[3]:
```
```python
# Plain-English summary of identified faults (based on the previously computed results)
summary = (
    "Fault identification results (window 2020-05-29 02:00:00 to 02:30:00 Asia/Shanghai):\n"
    "- Total faults identified (after filtering): 216\n"
    "- Number of unique components with faults: 40\n"
    "- Earliest fault start timestamp: 2020-05-29 02:00:00+0800\n\n"
    "Top example faults (by run length / severity):\n"
    "1) os_021 / Disk_avgqu_sz: 3-minute run, max=16.74747, P95=0.574488, breach_ratio≈28.15\n"
    "2) os_022 / Disk_avgqu_sz: 3-minute run, max=18.46334, P95=1.227996, breach_ratio≈14.04\n"
    "3) os_022 / Disk_io_util: 3-minute run, max=24.33741, P95=4.25, breach_ratio≈4.73\n\n"
    "Notes: Each fault run is a consecutive sequence of anomalous points (value > group's global P95). "
    "Runs were kept if they lasted ≥2 minutes or exceeded P95 by ≥50%."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                    kpi_name     fault_start_timestamp       fault_end_timestamp  run_length_minutes  max_value_in_run  global_P95_value  breach_ratio
0      os_021               Disk_avgqu_sz  2020-05-29 02:18:00+0800  2020-05-29 02:20:00+0800                   3      1.674747e+01      5.744880e-01     28.152001
1      os_022               Disk_avgqu_sz  2020-05-29 02:18:00+0800  2020-05-29 02:20:00+0800                   3      1.846334e+01      1.227996e+00     14.035338
2      os_022                Disk_io_util  2020-05-29 02:18:00+0800  2020-05-29 02:20:00+0800                   3      2.433741e+01      4.250000e+00      4.726449
3      os_013             CPU_iowait_time  2020-05-29 02:05:00+0800  2020-05-29 02:07:00+0800                   3      5.642810e-01      1.531810e-01      2.683746
4      os_022             CPU_iowait_time  2020-05-29 02:18:00+0800  2020-05-29 02:20:00+0800                   3      3.443786e+00      1.116197e+00      2.085285
5      os_013                  Disk_svctm  2020-05-29 02:08:00+0800  2020-05-29 02:10:00+0800                   3      5.850000e+00      2.093500e+00      1.794364
6      os_011    Outgoing_network_traffic  2020-05-29 02:08:00+0800  2020-05-29 02:10:00+0800                   3      2.886100e-02      1.035100e-02      1.788285
7      os_021                Disk_io_util  2020-05-29 02:19:00+0800  2020-05-29 02:21:00+0800                   3      2.100000e+01      1.186899e+01      0.769316
8      os_015        Processor_load_5_min  2020-05-29 02:12:00+0800  2020-05-29 02:14:00+0800                   3      2.333300e-02      1.858300e-02      0.255607
9   redis_011                used_cpu_sys  2020-05-29 02:01:00+0800  2020-05-29 02:03:00+0800                   3      7.800000e-01      7.010000e-01      0.112696
10     os_014                 Disk_wr_kbs  2020-05-29 02:07:00+0800  2020-05-29 02:09:00+0800                   3      5.823100e-02      5.348500e-02      0.088741
11     os_018       Processor_load_15_min  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      3.250000e-02      3.010400e-02      0.079585
12     db_005                    MEM_Used  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      4.301294e+02      4.300750e+02      0.000126
13  redis_012                 used_memory  2020-05-29 02:04:00+0800  2020-05-29 02:06:00+0800                   3      2.739840e+09      2.739666e+09      0.000063
14  redis_012  total_connections_received  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      2.369275e+07      2.369147e+07      0.000054
15  redis_001               keyspace_hits  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      1.062244e+10      1.062209e+10      0.000033
16  redis_001             keyspace_misses  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      8.051670e+09      8.051412e+09      0.000032
17  redis_002               keyspace_hits  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      1.052750e+10      1.052717e+10      0.000032
18  redis_002             keyspace_misses  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      8.217914e+09      8.217651e+09      0.000032
19  redis_009               keyspace_hits  2020-05-29 02:27:00+0800  2020-05-29 02:29:00+0800                   3      1.042125e+10      1.042092e+10      0.000032, 216, 40, '2020-05-29 02:00:00+0800')```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing trace_df if available, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

shanghai = pytz.timezone('Asia/Shanghai')

# ensure dt column exists and is timezone-aware Asia/Shanghai
if 'dt' not in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)
else:
    if not pd.api.types.is_datetime64tz_dtype(trace_df['dt']):
        trace_df['dt'] = pd.to_datetime(trace_df['dt'], utc=True).dt.tz_convert(shanghai)

# Define incident window in Asia/Shanghai
window_start = pd.Timestamp('2020-05-29 02:00:00', tz=shanghai)
window_end = pd.Timestamp('2020-05-29 02:30:00', tz=shanghai)

# 1) Compute global P95 per (cmdb_id, trace_name) over full series (before filtering)
p95_df = trace_df.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_P95_value'})

# total_points per group over full series
total_points_df = trace_df.groupby(['cmdb_id','trace_name'], as_index=False).size().rename(columns={'size':'total_points'})

# 2) Filter to incident window and sort
window_trace = trace_df[(trace_df['dt'] >= window_start) & (trace_df['dt'] <= window_end)].copy()
window_trace = window_trace.sort_values(['cmdb_id','trace_name','dt']).reset_index(drop=True)

# 3) Merge global P95 into window points and mark anomalies
window_with_p95 = window_trace.merge(p95_df, on=['cmdb_id','trace_name'], how='left')
window_with_p95['is_anomaly'] = window_with_p95['value'] > window_with_p95['global_P95_value']

# 4) Aggregate per group for window-level stats
window_agg = window_with_p95.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    points_in_window = ('value','size'),
    anomaly_count_in_window = ('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window = ('dt', lambda s: s[window_with_p95.loc[s.index,'is_anomaly']].min() if s.size>0 else pd.NaT),
    max_value_in_window = ('value','max')
)

# 5) Combine full-series totals and P95 with window aggregates to produce final per-group table
trace_groups = total_points_df.merge(p95_df, on=['cmdb_id','trace_name'], how='left').merge(
    window_agg, on=['cmdb_id','trace_name'], how='left'
)

# fill missing window-derived fields
trace_groups['points_in_window'] = trace_groups['points_in_window'].fillna(0).astype(int)
trace_groups['anomaly_count_in_window'] = trace_groups['anomaly_count_in_window'].fillna(0).astype(int)
trace_groups['max_value_in_window'] = trace_groups['max_value_in_window']  # keep NaN if no points
# format earliest timestamp: None if NaT
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(shanghai).strftime('%Y-%m-%d %H:%M:%S%z')
trace_groups['earliest_anomaly_timestamp_in_window'] = trace_groups['earliest_anomaly_timestamp_in_window'].apply(fmt_ts)

# Round numeric columns for compactness
trace_groups['global_P95_value'] = trace_groups['global_P95_value'].round(6)
trace_groups['max_value_in_window'] = trace_groups['max_value_in_window'].round(6)

# 6) Top trace groups table: sort by anomaly_count_in_window desc then points_in_window desc, limit 20
top_trace_groups = trace_groups.sort_values(['anomaly_count_in_window','points_in_window'], ascending=[False, False]).reset_index(drop=True)
top_trace_groups = top_trace_groups[['cmdb_id','trace_name','total_points','points_in_window','global_P95_value',
                                     'anomaly_count_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window']].head(20)

# 7) Focused table for trace_name containing 'os_022' or 'os_021' (case-sensitive)
mask_focus = trace_groups['trace_name'].str.contains('os_022|os_021', na=False)
focused_trace_groups = trace_groups[mask_focus].copy()
focused_trace_groups = focused_trace_groups.sort_values(['anomaly_count_in_window','points_in_window'], ascending=[False, False])
focused_trace_groups = focused_trace_groups[['cmdb_id','trace_name','total_points','points_in_window','global_P95_value',
                                             'anomaly_count_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window']].head(20)

# 8) Scalar: total number of trace anomaly points in window (sum of anomaly_count_in_window across all groups)
total_number_of_trace_anomaly_points_in_window = int(trace_groups['anomaly_count_in_window'].sum())

# Display compact results
top_trace_groups, focused_trace_groups, total_number_of_trace_anomaly_points_in_window

```
Out[4]:
```
```python
summary = (
    "Trace anomaly summary (window 2020-05-29 02:00:00 to 02:30:00 Asia/Shanghai):\n\n"
    "- Total trace anomaly points in the window: 271.\n\n"
    "- Top trace groups (examples):\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.duration_mean: total_points=30, points_in_window=30, "
    "global_P95=0.464565, anomaly_count=2, earliest_anomaly=2020-05-29 02:05:00+0800, max_in_window=0.505328.\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.duration_p95: total_points=30, points_in_window=30, "
    "global_P95=1.619310, anomaly_count=2, earliest_anomaly=2020-05-29 02:05:00+0800, max_in_window=1.803000.\n"
    "  * docker_001 / trace.self.csf.duration_mean and related self/row_count/jdbc/local traces also show 2 anomalies each (30 points in-window).\n\n"
    "- Focused groups containing 'os_022' or 'os_021' (examples):\n"
    "  * Several containers (docker_001, docker_002, docker_003, docker_004) have traces from os_022 or os_021 with the same pattern: "
    "30 points in-window, global P95 around 0.42–0.50 for duration_mean traces, anomaly_count=2, earliest anomalies typically between 02:00 and 02:05, "
    "and max values slightly above the P95.\n\n"
    "Interpretation: anomalies are present but sparse per group (mostly 2 anomalous minutes per group across the 30-minute window). "
    "They appear across many trace groups and several containers, including traces referencing os_022 and os_021. "
    "Total anomalous trace points across all groups in the incident window = 271."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                       trace_name  total_points  points_in_window  global_P95_value  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
0   docker_001    trace.from_os_022.remoteprocess.duration_mean            30                30          0.464565                        2             2020-05-29 02:05:00+0800             0.505328
1   docker_001     trace.from_os_022.remoteprocess.duration_p95            30                30          1.619310                        2             2020-05-29 02:05:00+0800             1.803000
2   docker_001        trace.from_os_022.remoteprocess.row_count            30                30        136.500000                        2             2020-05-29 02:01:00+0800           146.000000
3   docker_001                     trace.self.csf.duration_mean            30                30          0.048389                        2             2020-05-29 02:05:00+0800             0.054124
4   docker_001                      trace.self.csf.duration_p95            30                30          0.117485                        2             2020-05-29 02:00:00+0800             0.179450
5   docker_001                         trace.self.csf.row_count            30                30        546.000000                        2             2020-05-29 02:01:00+0800           584.000000
6   docker_001               trace.self.flyremote.duration_mean            30                30          0.009655                        2             2020-05-29 02:00:00+0800             0.009913
7   docker_001                trace.self.flyremote.duration_p95            30                30          0.016902                        2             2020-05-29 02:00:00+0800             0.057100
8   docker_001                   trace.self.flyremote.row_count            30                30        136.500000                        2             2020-05-29 02:01:00+0800           146.000000
9   docker_001                    trace.self.jdbc.duration_mean            30                30          0.003168                        2             2020-05-29 02:05:00+0800             0.003501
10  docker_001                        trace.self.jdbc.row_count            30                30       2184.450000                        2             2020-05-29 02:01:00+0800          2338.000000
11  docker_001                   trace.self.local.duration_mean            30                30          0.010710                        2             2020-05-29 02:05:00+0800             0.013891
12  docker_001                    trace.self.local.duration_p95            30                30          0.090650                        2             2020-05-29 02:05:00+0800             0.095000
13  docker_001                       trace.self.local.row_count            30                30       1911.000000                        2             2020-05-29 02:01:00+0800          2044.000000
14  docker_001  trace.to_docker_007.remoteprocess.duration_mean            30                30          0.028533                        2             2020-05-29 02:14:00+0800             0.029326
15  docker_001   trace.to_docker_007.remoteprocess.duration_p95            30                30          0.040358                        2             2020-05-29 02:14:00+0800             0.043250
16  docker_001      trace.to_docker_007.remoteprocess.row_count            30                30        281.650000                        2             2020-05-29 02:01:00+0800           293.000000
17  docker_001  trace.to_docker_008.remoteprocess.duration_mean            30                30          0.026353                        2             2020-05-29 02:09:00+0800             0.026941
18  docker_001   trace.to_docker_008.remoteprocess.duration_p95            30                30          0.036797                        2             2020-05-29 02:09:00+0800             0.039000
19  docker_001      trace.to_docker_008.remoteprocess.row_count            30                30        271.550000                        2             2020-05-29 02:03:00+0800           292.000000,        cmdb_id                                     trace_name  total_points  points_in_window  global_P95_value  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
0   docker_001  trace.from_os_022.remoteprocess.duration_mean            30                30          0.464565                        2             2020-05-29 02:05:00+0800             0.505328
1   docker_001   trace.from_os_022.remoteprocess.duration_p95            30                30          1.619310                        2             2020-05-29 02:05:00+0800             1.803000
3   docker_001      trace.from_os_022.remoteprocess.row_count            30                30        136.500000                        2             2020-05-29 02:01:00+0800           146.000000
28  docker_002  trace.from_os_022.remoteprocess.duration_mean            30                30          0.428988                        2             2020-05-29 02:01:00+0800             0.446800
29  docker_002   trace.from_os_022.remoteprocess.duration_p95            30                30          1.346982                        2             2020-05-29 02:01:00+0800             1.496200
31  docker_002      trace.from_os_022.remoteprocess.row_count            30                30        130.550000                        2             2020-05-29 02:03:00+0800           133.000000
60  docker_003  trace.from_os_021.remoteprocess.duration_mean            30                30          0.504096                        2             2020-05-29 02:20:00+0800             0.602289
61  docker_003   trace.from_os_021.remoteprocess.duration_p95            30                30          1.758880                        2             2020-05-29 02:11:00+0800             1.951000
63  docker_003      trace.from_os_021.remoteprocess.row_count            30                30        132.550000                        2             2020-05-29 02:04:00+0800           145.000000
84  docker_004  trace.from_os_021.remoteprocess.duration_mean            30                30          0.503786                        2             2020-05-29 02:02:00+0800             0.545429
85  docker_004   trace.from_os_021.remoteprocess.duration_p95            30                30          1.588295                        2             2020-05-29 02:06:00+0800             1.702000
87  docker_004      trace.from_os_021.remoteprocess.row_count            30                30        137.900000                        2             2020-05-29 02:01:00+0800           149.000000
2   docker_001     trace.from_os_022.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000
30  docker_002     trace.from_os_022.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000
62  docker_003     trace.from_os_021.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000
86  docker_004     trace.from_os_021.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000, 271)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing dataframes if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

shanghai = pytz.timezone('Asia/Shanghai')

# Ensure dt columns exist and are timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)
else:
    if not pd.api.types.is_datetime64tz_dtype(metric_df['dt']):
        metric_df['dt'] = pd.to_datetime(metric_df['dt'], utc=True).dt.tz_convert(shanghai)

if 'dt' not in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(shanghai)
else:
    if not pd.api.types.is_datetime64tz_dtype(trace_df['dt']):
        trace_df['dt'] = pd.to_datetime(trace_df['dt'], utc=True).dt.tz_convert(shanghai)

# Incident window (Asia/Shanghai)
window_start = pd.Timestamp('2020-05-29 02:00:00', tz=shanghai)
window_end = pd.Timestamp('2020-05-29 02:30:00', tz=shanghai)

# ---------- Part 1: Recompute faults for metric.csv ----------
# 1) Global P95 per (cmdb_id, kpi_name) over full series (before filtering)
p95_metric = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_P95_value'})

# 2) Filter metric points to the incident window and sort
metric_window = metric_df[(metric_df['dt'] >= window_start) & (metric_df['dt'] <= window_end)].copy()
metric_window = metric_window.sort_values(['cmdb_id','kpi_name','dt']).reset_index(drop=True)

# 3) Merge p95 and mark anomalies
metric_window = metric_window.merge(p95_metric, on=['cmdb_id','kpi_name'], how='left')
metric_window['is_anomaly'] = metric_window['value'] > metric_window['global_P95_value']

# 4) Function to extract consecutive anomaly runs within a group's sorted slice
def extract_runs_from_group(g):
    # g sorted by dt
    if g.empty:
        return pd.DataFrame()
    mask = g['is_anomaly'].astype(int)
    # segment id increments whenever mask changes (within this group)
    seg = (mask != mask.shift(fill_value=0)).cumsum()
    g = g.assign(_seg=seg)
    anomalous = g[g['is_anomaly']]
    if anomalous.empty:
        return pd.DataFrame()
    agg = anomalous.groupby('_seg').agg(
        cmdb_id = ('cmdb_id','first'),
        kpi_name = ('kpi_name','first'),
        fault_start_timestamp = ('dt','min'),
        fault_end_timestamp = ('dt','max'),
        run_length_minutes = ('dt', 'count'),  # count of points in run
        max_value_in_run = ('value','max'),
        global_P95_value = ('global_P95_value','first')
    ).reset_index(drop=True)
    return agg

# 4) Apply per group
runs = []
for _, g in metric_window.groupby(['cmdb_id','kpi_name'], sort=False):
    r = extract_runs_from_group(g)
    if not r.empty:
        runs.append(r)
if runs:
    runs_metric_df = pd.concat(runs, ignore_index=True)
else:
    runs_metric_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'run_length_minutes','max_value_in_run','global_P95_value'
    ])

# 4.5) Compute breach_ratio
runs_metric_df['global_P95_value'] = runs_metric_df['global_P95_value'].astype(float)
runs_metric_df['max_value_in_run'] = runs_metric_df['max_value_in_run'].astype(float)
runs_metric_df['breach_ratio'] = (runs_metric_df['max_value_in_run'] - runs_metric_df['global_P95_value']) / runs_metric_df['global_P95_value'].clip(lower=1e-9)

# 5) Filter runs: run_length_minutes >=2 OR breach_ratio >= 0.5
filtered_runs_metric = runs_metric_df[(runs_metric_df['run_length_minutes'] >= 2) | (runs_metric_df['breach_ratio'] >= 0.5)].copy()

# Format timestamps and numeric columns for compact display
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(shanghai).strftime('%Y-%m-%d %H:%M:%S%z')

if not filtered_runs_metric.empty:
    filtered_runs_metric['fault_start_timestamp'] = filtered_runs_metric['fault_start_timestamp'].apply(fmt_ts)
    filtered_runs_metric['fault_end_timestamp'] = filtered_runs_metric['fault_end_timestamp'].apply(fmt_ts)
    filtered_runs_metric['global_P95_value'] = filtered_runs_metric['global_P95_value'].round(6)
    filtered_runs_metric['max_value_in_run'] = filtered_runs_metric['max_value_in_run'].round(6)
    filtered_runs_metric['breach_ratio'] = filtered_runs_metric['breach_ratio'].round(6)

# 1) Filter to only cmdb_id in {os_021, os_022}
nodes_of_interest = {'os_021','os_022'}
faults_os = filtered_runs_metric[filtered_runs_metric['cmdb_id'].isin(nodes_of_interest)].copy()

# Prepare compact display table (limit 20 rows)
faults_os_display = faults_os[['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                              'run_length_minutes','max_value_in_run','global_P95_value','breach_ratio']].head(20)

# Scalars: count of faults per node and earliest fault start by node
count_of_faults_os_021 = int(faults_os[faults_os['cmdb_id']=='os_021'].shape[0])
count_of_faults_os_022 = int(faults_os[faults_os['cmdb_id']=='os_022'].shape[0])

if not faults_os.empty:
    earliest_by_node = faults_os.groupby('cmdb_id', as_index=False)['fault_start_timestamp'].min()
    # ensure consistent formatting
    earliest_by_node = earliest_by_node.rename(columns={'fault_start_timestamp':'earliest_fault_start_timestamp'})
else:
    earliest_by_node = pd.DataFrame(columns=['cmdb_id','earliest_fault_start_timestamp'])

# ---------- Part 2: Trace aggregation focused on trace_name containing 'os_021' or 'os_022' ----------
# 1) Compute full-series global P95 per (cmdb_id, trace_name)
p95_trace = trace_df.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_P95_value'})
total_points_trace = trace_df.groupby(['cmdb_id','trace_name'], as_index=False).size().rename(columns={'size':'total_points'})

# 2) Filter trace points to window and merge p95
trace_window = trace_df[(trace_df['dt'] >= window_start) & (trace_df['dt'] <= window_end)].copy()
trace_window = trace_window.sort_values(['cmdb_id','trace_name','dt']).reset_index(drop=True)
trace_window = trace_window.merge(p95_trace, on=['cmdb_id','trace_name'], how='left')
trace_window['is_anomaly'] = trace_window['value'] > trace_window['global_P95_value']

# 3) Aggregate per group for window-level stats
trace_window_agg = trace_window.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    points_in_window = ('value','size'),
    anomaly_count_in_window = ('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window = ('dt', lambda s: s[trace_window.loc[s.index,'is_anomaly']].min() if s.size>0 else pd.NaT),
    max_value_in_window = ('value','max')
)

# 4) Combine with total_points and p95 to form full table
trace_groups = total_points_trace.merge(p95_trace, on=['cmdb_id','trace_name'], how='left').merge(
    trace_window_agg, on=['cmdb_id','trace_name'], how='left'
)

trace_groups['points_in_window'] = trace_groups['points_in_window'].fillna(0).astype(int)
trace_groups['anomaly_count_in_window'] = trace_groups['anomaly_count_in_window'].fillna(0).astype(int)
trace_groups['max_value_in_window'] = trace_groups['max_value_in_window']
trace_groups['earliest_anomaly_timestamp_in_window'] = trace_groups['earliest_anomaly_timestamp_in_window'].apply(lambda x: fmt_ts(x) if not pd.isna(x) else None)
trace_groups['global_P95_value'] = trace_groups['global_P95_value'].round(6)
trace_groups['max_value_in_window'] = trace_groups['max_value_in_window'].round(6)

# 5) Filter to trace_name containing 'os_021' or 'os_022' (case-sensitive)
focus_mask = trace_groups['trace_name'].str.contains('os_021|os_022', na=False)
trace_focused = trace_groups[focus_mask].copy()

trace_focused_display = trace_focused[['cmdb_id','trace_name','total_points','points_in_window','global_P95_value',
                                       'anomaly_count_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window']].head(20)

# Scalar: earliest_trace_anomaly_timestamp_among_these (minimum non-null earliest_anomaly_timestamp_in_window)
if trace_focused['earliest_anomaly_timestamp_in_window'].dropna().any():
    earliest_trace_anomaly_timestamp_among_these = trace_focused['earliest_anomaly_timestamp_in_window'].dropna().min()
else:
    earliest_trace_anomaly_timestamp_among_these = None

# Final compact outputs
faults_os_display, count_of_faults_os_021, count_of_faults_os_022, earliest_by_node, trace_focused_display, earliest_trace_anomaly_timestamp_among_these

```
Out[5]:
```
Part 1 — Metric fault runs (filtered to nodes os_021 and os_022)
- Counts:
  - Fault runs for os_021: 16
  - Fault runs for os_022: 7
- Earliest fault start by node:
  - os_021: 2020-05-29 02:00:00+0800
  - os_022: 2020-05-29 02:00:00+0800
- Notes / examples (top/representative runs found):
  - os_021 / Disk_avgqu_sz: 2020-05-29 02:18:00–02:20:00, run_length=3 min, max=16.747472, P95=0.574488, breach_ratio≈28.15
  - os_022 / Disk_avgqu_sz: 2020-05-29 02:18:00–02:20:00, run_length=3 min, max=18.463343, P95=1.227996, breach_ratio≈14.04
  - os_021 / CPU_iowait_time: runs at 02:18 (1 min) and 02:20–02:21 (2 min), large breach ratios (many× P95)
  - os_022 / Disk_io_util and CPU_system_time show multi-minute runs with substantial breaches
- General observation: faults on these two nodes are short (mostly 1–3 minute runs) but some runs show very large breach ratios (many times the group's P95), especially for disk and CPU I/O-related KPIs.

Part 2 — Trace groups referencing os_021 or os_022 (window 02:00–02:30 Asia/Shanghai)
- The focused trace table contains traces such as:
  - docker_001 / trace.from_os_022.remoteprocess.duration_mean: total_points=30, points_in_window=30, global_P95=0.464565, anomaly_count=2, earliest_anomaly=2020-05-29 02:05:00+0800, max_in_window=0.505328
  - docker_002 / trace.from_os_022.remoteprocess.duration_mean and similar duration_p95/row_count traces: 30 points in-window, anomaly_count typically = 2
  - docker_003/docker_004 have traces from os_021 with similar patterns (30 points in-window, 2 anomalous minutes)
- Earliest trace anomaly among these focused traces: 2020-05-29 02:01:00+0800
- General observation: traces referencing os_021/os_022 show sparse anomalies (usually 2 anomalous minutes per trace group over the 30-minute window), with earliest trace anomalies around 02:01–02:05.

If you want, I can present the compact metric-fault table and the compact trace table again (limited rows) for direct inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id               kpi_name     fault_start_timestamp       fault_end_timestamp  run_length_minutes  max_value_in_run  global_P95_value  breach_ratio
762  os_021        CPU_iowait_time  2020-05-29 02:18:00+0800  2020-05-29 02:18:00+0800                   1          3.048940          0.066402     44.916457
763  os_021        CPU_iowait_time  2020-05-29 02:20:00+0800  2020-05-29 02:21:00+0800                   2          4.017785          0.066402     59.507079
764  os_021        CPU_system_time  2020-05-29 02:00:00+0800  2020-05-29 02:00:00+0800                   1          0.705191          0.345163      1.043064
766  os_021          CPU_user_time  2020-05-29 02:00:00+0800  2020-05-29 02:00:00+0800                   1         14.948100          4.422556      2.379968
770  os_021          Disk_avgqu_sz  2020-05-29 02:18:00+0800  2020-05-29 02:20:00+0800                   3         16.747472          0.574488     28.152001
771  os_021             Disk_await  2020-05-29 02:28:00+0800  2020-05-29 02:29:00+0800                   2          3.520000          0.560500      5.280107
772  os_021           Disk_io_util  2020-05-29 02:19:00+0800  2020-05-29 02:21:00+0800                   3         21.000000         11.868991      0.769316
773  os_021            Disk_rd_ios  2020-05-29 02:06:00+0800  2020-05-29 02:06:00+0800                   1          1.000000          0.100000      9.000000
774  os_021            Disk_rd_ios  2020-05-29 02:27:00+0800  2020-05-29 02:27:00+0800                   1          1.000000          0.100000      9.000000
775  os_021            Disk_rd_kbs  2020-05-29 02:06:00+0800  2020-05-29 02:06:00+0800                   1          0.006565          0.000289     21.724126
776  os_021            Disk_rd_kbs  2020-05-29 02:27:00+0800  2020-05-29 02:27:00+0800                   1          0.006068          0.000289     20.003808
792  os_021                Page_pi  2020-05-29 02:06:00+0800  2020-05-29 02:06:00+0800                   1          6.703926          0.297481     21.535618
793  os_021                Page_pi  2020-05-29 02:27:00+0800  2020-05-29 02:27:00+0800                   1          6.311049          0.297481     20.214940
797  os_021  Processor_load_15_min  2020-05-29 02:28:00+0800  2020-05-29 02:29:00+0800                   2          0.097812          0.084109      0.162920
800  os_021   Processor_load_1_min  2020-05-29 02:28:00+0800  2020-05-29 02:28:00+0800                   1          0.203437          0.126578      0.607211
802  os_021   Processor_load_5_min  2020-05-29 02:28:00+0800  2020-05-29 02:29:00+0800                   2          0.137187          0.105719      0.297660
809  os_022        CPU_iowait_time  2020-05-29 02:18:00+0800  2020-05-29 02:20:00+0800                   3          3.443786          1.116197      2.085285
810  os_022        CPU_system_time  2020-05-29 02:00:00+0800  2020-05-29 02:01:00+0800                   2          0.624010          0.362395      0.721903
811  os_022          CPU_user_time  2020-05-29 02:00:00+0800  2020-05-29 02:00:00+0800                   1         14.490963          5.662677      1.559031
815  os_022          Disk_avgqu_sz  2020-05-29 02:18:00+0800  2020-05-29 02:20:00+0800                   3         18.463343          1.227996     14.035338, 16, 7,   cmdb_id earliest_fault_start_timestamp
0  os_021       2020-05-29 02:00:00+0800
1  os_022       2020-05-29 02:00:00+0800,        cmdb_id                                     trace_name  total_points  points_in_window  global_P95_value  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window
0   docker_001  trace.from_os_022.remoteprocess.duration_mean            30                30          0.464565                        2             2020-05-29 02:05:00+0800             0.505328
1   docker_001   trace.from_os_022.remoteprocess.duration_p95            30                30          1.619310                        2             2020-05-29 02:05:00+0800             1.803000
2   docker_001     trace.from_os_022.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000
3   docker_001      trace.from_os_022.remoteprocess.row_count            30                30        136.500000                        2             2020-05-29 02:01:00+0800           146.000000
28  docker_002  trace.from_os_022.remoteprocess.duration_mean            30                30          0.428988                        2             2020-05-29 02:01:00+0800             0.446800
29  docker_002   trace.from_os_022.remoteprocess.duration_p95            30                30          1.346982                        2             2020-05-29 02:01:00+0800             1.496200
30  docker_002     trace.from_os_022.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000
31  docker_002      trace.from_os_022.remoteprocess.row_count            30                30        130.550000                        2             2020-05-29 02:03:00+0800           133.000000
60  docker_003  trace.from_os_021.remoteprocess.duration_mean            30                30          0.504096                        2             2020-05-29 02:20:00+0800             0.602289
61  docker_003   trace.from_os_021.remoteprocess.duration_p95            30                30          1.758880                        2             2020-05-29 02:11:00+0800             1.951000
62  docker_003     trace.from_os_021.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000
63  docker_003      trace.from_os_021.remoteprocess.row_count            30                30        132.550000                        2             2020-05-29 02:04:00+0800           145.000000
84  docker_004  trace.from_os_021.remoteprocess.duration_mean            30                30          0.503786                        2             2020-05-29 02:02:00+0800             0.545429
85  docker_004   trace.from_os_021.remoteprocess.duration_p95            30                30          1.588295                        2             2020-05-29 02:06:00+0800             1.702000
86  docker_004     trace.from_os_021.remoteprocess.error_rate            30                30          0.000000                        0                                 None             0.000000
87  docker_004      trace.from_os_021.remoteprocess.row_count            30                30        137.900000                        2             2020-05-29 02:01:00+0800           149.000000, '2020-05-29 02:01:00+0800')```
```